# Classification Basics

Classification is one of the most useful and popular functions in data mining and machine learning. 

Essentially, classification is aimed at building a **prediction model** that can assign data points to a set of **predefined classes**, i.e., giving a class label to each data point. 

There are several key concepts related to classification: 
- **Prediction model**: There is a specific target to predict. 
- **Class label**: the target variable is qualitative/categorical.
- **Predefined classes**: unlike clustering, classification has predefined classes. 
    - Binary classification: 2 classes (Yes vs. No, True vs. False, Positive vs. Negative)
    - Multi-class classification: 3 or more classes
- **Supervised learning**: the predefined classes supervise the learning process.
- **Learning what?** 
    - The **function** between predictors X and class label L. 
    - F(X) -> L
- **Learning from what?**
    - **Training set**: $(X_{train}, L_{train})$
    - $X_{train}$: predictor variables of data points in the training set
    - $L_{train}$: class labels of data points in the training set
- **How accurate is the model?**
    - **Test set**: $(X_{test}, L_{test})$ 
    - $X_{test}$: predictor variables of data points in the test set
    - Apply the model to $X_{test}$ and output $L'_{test}$, **predicted** labels. 
    - $L_{test}$: **actual** labels of data points in the test set, hidden from the model
    - Evaluation by comparing $L_{test}$ and $L'_{test}$
- **Evaluation metrics**: 
    - Accuracy: 0~100%
    - Error rate: 1-accuracy
    - Confusion matrix: 
        - Type I error: false positives (FP)
        - Type II error: false negatives (FN)
    - Sensitivity, specificity, precision, recall, F-measure, AUC...

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Colab only: mount Google Drive and set output folder for submissions
from google.colab import drive

drive.mount('/content/drive')

output_dir = '/content/drive/My Drive/Colab Notebooks/output/' # You can choose your own output folder
os.makedirs(output_dir, exist_ok=True)
print(f'Submission files will be saved to: {output_dir}')

## Titanic Data 

We will use the famous Titanic data to illustrate the process of classification. 

In [ ]:
# Load the training set
df_train = pd.read_csv('https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/data/titanic_train.csv')
df_train.info()

In [ ]:
s = df_train['Survived'].value_counts()
print(s)
print(f'the survival rate is {s[1]/s.sum():.1%}')

In [ ]:
s_norm = df_train['Survived'].value_counts(normalize=True)
s_norm

## A Naive Prediction Model

Simply based on the overall survival rate (38.4%), we could build a naive prediction model: 
- Survived = 0 (dead) for all

Let's apply this model to the test set. 

In [ ]:
df_test = pd.read_csv('https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/data/titanic_test.csv')
df_test.info()

In [ ]:
# predict survived = 0 (dead) for all because survival rate is 38.4%
df_test['Survived'] = 0
df_test.head()

In [ ]:
# Select the two columns and save to a .csv file for submission
df_submit_allzero = df_test[['PassengerId', 'Survived']]
# Set index=False to exclude the index column
df_submit_allzero.to_csv(f'{output_dir}/titanic_submit_allzero.csv', index=False)

Submit this file "titanic_submit_allzero.csv" to Kaggle. Check your score and ranking. 

Can we do better than this? 

In [ ]:
# Consider Sex as a predictor variable
group = df_train.groupby('Sex')['Survived'].value_counts()
print(group.index)
group

In [ ]:
print(f'female survival rate: {group["female", 1]/group["female"].sum():.1%}')
print(f'male survival rate: {group["male", 1]/group["male"].sum():.1%}')

In [ ]:
# another (easier) way to calculate the survival rates by gender
group_norm = df_train.groupby('Sex')['Survived'].value_counts(normalize=True)
group_norm

## Another Simple Prediction Model

Since female survival rate is 74.2% and male survival rate is 18.9%, we could build another simple prediction model: 
- Survived = 0 (dead) for male 
- Survived = 1 (survived) for female

In [ ]:
# predict survived = 0 (dead) for male and survived = 1 (survived) for female 
# because female has a higher survival rate (74.2%) than male (18.9%)
df_test['Survived_Gender'] = df_test['Sex'].apply(lambda x: 0 if x=='male' else 1)
df_test.head()

In [ ]:
df_submit_gender = df_test[['PassengerId', 'Survived_Gender']]
# change column name to 'Survived' for Kaggle submission
df_submit_gender.to_csv(f'{output_dir}/titanic_submit_gender.csv', index=False, header=['PassengerId', 'Survived']) 

Submit this file "titanic_submit_gender.csv" to Kaggle. Check your score and ranking. 

## Decision Tree for the Titanic Data

A decision tree is a prediction model that uses a tree-like structure of decisions and their possible consequences. It can be used for classification and regression. 

The basic idea of a decision tree is to split data set based on the homogeneity of data, i.e., reducing “impurity”. 

### Entropy

Entropy is one of the most common measures for calculating impurity.

$H(x)=-\sum_{i=1}^np_{i}\log p_{i}$, where $p_{i}$ is the probability of class $i$ in the data.

In [ ]:
# Entropy for a collection of 30 balls with 15 red and 15 blue
# Entropy = 1     i.e., maximal impurity 
e1 = -15/30*np.log2(15/30)-15/30*np.log2(15/30)
e1

In [ ]:
# Entropy for a collection of 30 balls with 2 red and 28 blue
e2 = -2/30*np.log2(2/30)-28/30*np.log2(28/30)
e2

In [ ]:
# Entropy for a collection of 30 balls with 0 red and 30 blue
# Entropy = 0     i.e., maximal purity 
e3 = -np.log2(30/30)
e3

In [ ]:
# Define a function to calculate entropy
# input: a list of values that represents ratio of different classes, e.g., (1,2,3,4)
def entropy(ratio):
    h = None
    s = 0
    for i in ratio:
        s += i
    if s>0:
        h = 0
        for i in ratio: 
            if i>0:
                h += -i/s*np.log2(i/s)
    return h

In [ ]:
r = input('Please enter the ratio in format 1:2:3 : ')
ratio = [int(i) for i in r.split(':')]
entropy(ratio)

In [ ]:
fig = plt.figure()
ax = plt.axes()
x = np.arange(0, 31, 1) # return numbers [0,1,2,...,30]
e = pd.Series([entropy([i,30-i]) for i in x])
ax.plot(x, e)
plt.show()

## scikit-learn

Scikit-learn is one of the most popular Python packages for predictive data analysis. 

https://scikit-learn.org/

We will use sklearn for building decision trees. 

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df_train.head()

In [ ]:
# sklearn decision tree only accept inputs in float data type without null values
# we will learn data processing and transformation later to handle null values and non-numerical features
# thus, we choose three numerical values to train the tree classifier
# pclass: Ticket class - 1 = 1st, 2 = 2nd, 3 = 3rd
# sibsp: of siblings / spouses aboard the Titanic
# fare: Passenger fare

X = df_train[['Pclass', 'SibSp', 'Fare']]
y = df_train['Survived']
print(X.shape)
print(y.shape)


In [ ]:
# Train a DT model by using all default settings. 
# The default criterion='gini', not 'entropy'. 
from sklearn.tree import DecisionTreeClassifier
tree_clf = DecisionTreeClassifier()
tree_clf.fit(X, y)

In [ ]:
# For visualizing the tree
# This may require you installing a package called "graphviz" as well.
from IPython.display import Image  
from sklearn import tree
import pydotplus

In [ ]:
# get feature and class names for visualization
print(X.columns.values.tolist())
print(y.unique().tolist())
cls_names = ['died' if i == 0 else 'survived' for i in y.unique().tolist()] # convert to string for class names
cls_names

In [ ]:
# Create DOT data
dot_data = tree.export_graphviz(tree_clf, 
                                feature_names=X.columns.values.tolist(),  
                                class_names=cls_names, 
                               )

# Draw graph
graph = pydotplus.graph_from_dot_data(dot_data)  

# Show graph
Image(graph.create_png())

You can see this is a huge tree, which may lead to *overfitting* problem. 

Let's set max_depth=3 to generate a simpler tree. Also, set criterion='entropy'.

In [ ]:
tree_clf = DecisionTreeClassifier(criterion='entropy', max_depth=3)  
tree_clf.fit(X, y)

In [ ]:
# Create DOT data
dot_data = tree.export_graphviz(tree_clf, 
                                feature_names=X.columns.values.tolist(),  
                                class_names=cls_names,
                               )

# Draw graph
graph = pydotplus.graph_from_dot_data(dot_data)  

# Show graph
Image(graph.create_png())

### About Each Node
- First line, such as Pclass<=2.5: the tree is split based on the true or false of the condition Pclass<=2.5
- Second line, such as entropy=0.961: the current entropy of the node
- Third line, such as samples = 891: the total number of samples in this node
- Fourth line, such as value = [549, 342]: how many samples belong to each class, 549 belong to class 0 (died) and 342 belong to class 1 (survived)
- Last line, such as class = died: this shows the prediction a given node will make and the class occurs the most within the node will be selected as the class prediction.

### Make Predictions

Remember the training data has three features ['Pclass', 'SibSp', 'Fare'], we can predict the target based on different values for those three features

In [ ]:
# passenger 1 who bought a class 3 ticket at $8.5 with no siblings / spouses, Jack ? 
# passenger 2 who bought a first class ticket at $88 with no siblings / spouses, Rose ?
passenger1 = tree_clf.predict([[3, 0, 8.5]])
passenger2 = tree_clf.predict([[1, 0, 88]])

print(passenger1, passenger2)

Next, we use this simple DT to predict for the test set. 

In [ ]:
# choose the subset of the test set
# there is a null value in Fare
X_test = df_test[['Pclass', 'SibSp', 'Fare']]
X_test.info()

In [ ]:
# fill the null value with mean
X_test = X_test.fillna(X_test.mean())
X_test.info()

In [ ]:
y_hat = tree_clf.predict(X_test)
y_hat

In [ ]:
# combine the final dataframe
df_submit_simpleDT = pd.DataFrame({
                                    'PassengerId': df_test['PassengerId'], 
                                    'Survived': y_hat,
                                  })
df_submit_simpleDT.head()

In [ ]:
df_submit_simpleDT.to_csv(f'{output_dir}/titanic_submit_simpleDT.csv', index=False) 

Submit this file "titanic_submit_simpleDT.csv" to Kaggle. Check your score and ranking.